# SOLUTIONS · Laptop lab 4: Missing values, trees and forests
*Keep every listing, and let many trees decide*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 19, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab04_solutions.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you learned to fill the gaps in a table and to keep the features that matter (Lecture 7), and you met decision trees and random forests (Lecture 8). Today you use both on the Airbnb listings of Lab 3.

**The story.** The question is the same as in Lab 3: is a listing **premium**, more than €300 per night? But one thing has changed. In Lab 3 we quietly dropped every listing with a missing value, about one in six. Today they stay: a pipeline that stops at a gap cannot help a host whose listing has no reviews yet. As before, 2,000 other listings stay locked away, gaps included: the **test set**.

**Your mission.** Build one pipeline that classifies *every* listing as premium or not, gaps and all, and estimate its accuracy on new listings. The warm-up shows you the pieces; then you decide how to put them together.

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. A check below the cell shows an orange *Not yet* box until you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

Imputers and trees work like every other step in scikit-learn:

```python
imputer = SimpleImputer(strategy="median")    # fills every gap of a column with its median
imputer.fit(X_train)                          # learns one median per column
imputer.transform(X_new)                      # the gaps become those medians

tree = DecisionTreeClassifier(max_depth=3)    # a tree of at most 3 questions per path
tree.fit(X_train, y_train)
tree.predict(X_new)                           # the majority class of the leaf
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1–3. Warm-up: gaps, a tree, a forest | together on the screen, 15 |
| 4. Your mission | 45 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the tools of this lab and the data from Inside Airbnb, and cleans them as in Lab 1, but this time without dropping the listings with gaps. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# The tools of this lab are in the course lab repository;
# they also download the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab04_tools.py", "lab04_tools.py")
from lab04_tools import *

X, y = load_training_data()    # your 1,500 listings: features X, and y = 1 for premium
setup_ok()

**Run.** Your data: one row per listing, `y` = 1 if it costs more than €300 per night. `X` has three groups of columns, each in a list:

| List | Columns |
|---|---|
| `NUMERIC` | the 12 numbers of Lab 3: `accommodates`, `bedrooms`, `beds`, `bathrooms`, `minimum_nights`, `availability_365`, `number_of_reviews`, `reviews_per_month`, `review_scores_rating`, `host_listings`, `superhost`, `km_to_center` |
| `CATEGORICAL` | the 2 text columns of Lab 3: `room_type`, `neighbourhood` |
| `LOCATION` | `latitude`, `longitude`: where the listing is, on the map. `km_to_center` was made from them, by hand, in Lab 2 |

Some numbers are missing: in pandas, a gap is `NaN` ("not a number").

In [ ]:
print(X.shape, "  share of premium listings:", round(y.mean(), 2))
X.head()

## Warm-up: the pieces, one by one (Tasks 1 to 3)

We go through Tasks 1 to 3 together on the screen: five cells, just run them and follow along. You will reuse them in your mission.

## Task 1. The gaps

**Run.** Where are the gaps, and how many listings have at least one? (Lecture 7, "The gaps in the survey".)

In [ ]:
gaps = X[NUMERIC].isna().sum()
print(gaps[gaps > 0])
print()
print("listings with at least one gap:", X[NUMERIC].isna().any(axis=1).sum(), "of", len(X))

**Run.** Why is a value missing? Before you fill anything in, look at who has the gap (Lecture 7, "Why is it missing?"). Here: which listings have no review score?

In [ ]:
no_score = X["review_scores_rating"].isna()
pd.crosstab(no_score, X["number_of_reviews"] == 0,
            rownames=["no review score"], colnames=["no reviews at all"])

Every listing without a review score has no reviews at all: the gap **depends on another column**. That tells you how to fill it: a listing without reviews gets 0 reviews per month, not the median.

Two more things to keep in mind for the mission. Logistic regression stops at the first `NaN`, so an imputer has to fill the gaps, **inside the pipeline**, before the scaler (Lecture 7, "The imputer belongs in the pipeline"). And a gap can carry information of its own: `SimpleImputer(add_indicator=True)` keeps it as an extra 0/1 column "was missing" (Lecture 7, "The absence is informative").

## Task 2. A decision tree

**Run.** A tree of depth 2 on two features: the number of guests and the distance to the center. Read it from the top: every box asks one yes/no question; a listing goes left if the answer is yes (Lecture 8, "A tree asks yes/no questions").

In [ ]:
two = ["accommodates", "km_to_center"]
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X[two], y)

fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(tree, feature_names=two, class_names=["not premium", "premium"], filled=True, ax=ax)
plt.show()

In every box: the question, the Gini impurity (0 for a pure group, 0.5 for a fifty-fifty group), the number of listings (`samples`), how many of each class (`value`), and the majority class, which is what the tree predicts there.

**Run.** How deep should the tree be? Now on all features. A tree needs **no scaler**, since a threshold does not care about units, but it still needs numbers, so the text columns are one-hot encoded, and the gaps are filled. The grid search tries depths 1 to 15; `return_train_score=True` also keeps the training accuracy (Lecture 8, "Choosing the depth").

In [ ]:
prep_tree = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),   # fill; no scaler for a tree
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    verbose_feature_names_out=False)                                   # keep the plain column names
tree_pipe = make_pipeline(prep_tree, DecisionTreeClassifier(random_state=0))

depths = list(range(1, 16))
grid_tree = GridSearchCV(tree_pipe, {"decisiontreeclassifier__max_depth": depths}, cv=5, return_train_score=True)
grid_tree.fit(X, y)
print("best depth:", grid_tree.best_params_, "  mean validation accuracy:", round(grid_tree.best_score_, 3))

fig, ax = plt.subplots()
ax.plot(depths, grid_tree.cv_results_["mean_train_score"], marker="o", label="training")
ax.plot(depths, grid_tree.cv_results_["mean_test_score"], marker="o", label="validation (5-fold)")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

The picture of Lecture 3 again: the training accuracy keeps climbing towards 1, the validation accuracy stays flat and then drops. A deep tree memorizes the training listings. `prep_tree` and the grid search `grid_tree` are ready to reuse in your mission.

## Task 3. One tree against a forest

**Run.** Where are the premium apartments? We take only the whole homes and apartments, so that the room type cannot explain the price, and give the models nothing but the two coordinates, `latitude` and `longitude`. Then we draw what each model predicts on the map of Amsterdam: logistic regression, two single trees of a random forest, and the whole forest of 200 trees. Each tree is grown on its own bootstrap sample of the listings, and every question chooses among a random subset of the features; then the trees vote (Lecture 8, "Three trees and the forest"). The star is Dam Square.

In [ ]:
whole = (X["room_type"] == "Entire home/apt").to_numpy()      # whole homes and apartments only
X_home, y_home = X.loc[whole, LOCATION], y[whole]

loc_logreg = make_pipeline(StandardScaler(), LogisticRegression())
loc_forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=0, n_jobs=-1)
print("always the most common class:", round(max(y_home.mean(), 1 - y_home.mean()), 3))
for name, m in [("logistic regression", loc_logreg), ("random forest", loc_forest)]:
    print(f"{name:20} mean validation accuracy {cross_val_score(m, X_home, y_home, cv=5).mean():.3f}")
loc_logreg.fit(X_home, y_home)
loc_forest.fit(X_home, y_home)

lon = np.linspace(4.83, 4.97, 200)
lat = np.linspace(52.33, 52.41, 200)
LON, LAT = np.meshgrid(lon, lat)
city = pd.DataFrame({"latitude": LAT.ravel(), "longitude": LON.ravel()})
tree_1, tree_2 = loc_forest.estimators_[0], loc_forest.estimators_[1]     # two trees of the forest
panels = [("logistic regression", loc_logreg.predict_proba(city)[:, 1]),
          ("one tree of the forest", tree_1.predict_proba(city.to_numpy())[:, 1]),
          ("another tree", tree_2.predict_proba(city.to_numpy())[:, 1]),
          ("the forest: 200 trees vote", loc_forest.predict_proba(city)[:, 1])]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.3), sharey=True)
for ax, (name, p) in zip(axes, panels):
    filled = ax.contourf(LON, LAT, p.reshape(LON.shape), levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.8)
    ax.plot(4.8932, 52.3731, marker="*", color="black", markersize=14)     # Dam Square
    ax.set_title(name)
    ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
fig.colorbar(filled, ax=axes, label="probability of premium", shrink=0.8)
plt.show()

Logistic regression can only draw one straight line through the city, and from two coordinates it learns almost nothing: it scores about the same as always guessing the most common class. A single tree cuts the map into rectangles, and two trees of the same forest disagree in many places: a tree is unstable. The forest averages the 200 votes, and a picture appears: blue around the center and to the south, red to the west and north. Nobody told it where the center is.

In your mission, you build a forest on all the features and compare it with one tree.

## Task 4. Your mission

Now it is your turn. Build **one pipeline** that classifies every listing as premium or not, **gaps included**, and estimate its accuracy on the locked listings.

You do this in four short steps. Step 1 is ready to run. In Steps 2 to 4 the comments in each cell say what to do; copy from Tasks 1 to 3, from Step 1 and from the **toolbox** right below. After every run, ask yourself: did the mean validation accuracy go up or down?

**The rule:** you may do anything with `X` and `y`. The test set stays locked. Your pipeline must work for listings with gaps.

Write down your results in the notes cell at the end of the mission; you will need them to choose.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

### 1. Filling the gaps inside the pipeline

An imputer is one more step, before the scaler. In the column transformer, give the numbers a small pipeline: first fill, then scale.

```python
(make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC)
```

The choices (Lecture 7, "Fill in a constant" to "Neighbors and models"):

```python
SimpleImputer(strategy="median")                       # the median of the column: the default
SimpleImputer(strategy="median", add_indicator=True)   # also a 0/1 column "was missing" per column with gaps
SimpleImputer(strategy="constant", fill_value=0)       # a number you choose, from knowledge of the problem
make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5))   # the average of the 5 most similar listings
```

`KNNImputer` measures distances, so scale before it (`StandardScaler` simply skips the gaps). Different columns can get different fills: give each group its own line, as in Lab 3.

```python
zero_fill = ["reviews_per_month"]                       # no reviews: 0 per month
other = [c for c in NUMERIC if c not in zero_fill]

prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), zero_fill),
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), other),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
```

### 2. A decision tree, and its size

A tree needs no scaler, but it needs numbers and (in this lab) filled gaps:

```python
prep_tree = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    verbose_feature_names_out=False)

my_tree = make_pipeline(prep_tree, DecisionTreeClassifier(max_depth=4, random_state=0))
```

The size of the tree is its complexity, like `k` in $k$-NN and `C` in logistic regression. One parameter is enough; choose it with a grid search (Lecture 8, "Keep the tree small"):

```python
grid = GridSearchCV(my_tree, {"decisiontreeclassifier__max_depth": range(1, 16)}, cv=5)
# or: {"decisiontreeclassifier__min_samples_leaf": [1, 5, 10, 20, 50]}
```

### 3. A random forest

Change only the model: the trees vote. More trees never hurt, they only cost time; `max_features` (how many features each question may choose from) is the one setting worth tuning (Lecture 8, "How many trees? How many features per question?").

```python
my_forest = make_pipeline(prep_tree, RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1))

grid = GridSearchCV(my_forest, {"randomforestclassifier__max_features": ["sqrt", 0.2, 0.4, None]}, cv=5)
```

`"sqrt"` (the default) lets each question choose among $\sqrt{p}$ of the $p$ features; `0.2` among 20% of them; `None` among all of them, which makes every tree almost the same. `n_jobs=-1` uses all processors of the computer.

Which features does the forest use most? Fit it, then read its importances, with doubt (Lecture 8, "Read with doubt"):

```python
my_forest.fit(X, y)
pd.Series(my_forest[-1].feature_importances_, index=my_forest[0].get_feature_names_out()).sort_values().tail(10)
```

### 4. Choosing features

After one-hot encoding, your pipeline sees about 40 columns: the 12 numbers, a "was missing" column for each number with gaps, 4 room types and 22 neighbourhoods. A forest chooses among them by itself; a linear model may do better with fewer.

The three ways of Lecture 7, in plain words:

- **Score each column alone** (`SelectKBest`). For one column at a time, `f_classif` asks: are its averages for premium and for other listings far apart, compared with the spread inside each group? The `k` columns with the largest score are kept. Fast, but blind to combinations: a column that matters only together with another scores low.
- **Ask a linear model** (`SelectFromModel` with an L1 penalty). The L1 penalty, as in lasso (Lecture 4), pushes weak weights to exactly 0; the columns with a nonzero weight are kept. A smaller `C` means a stronger penalty and fewer columns. The weights are compared, so scale the numbers first.
- **Ask a forest** (`SelectFromModel` with a forest). Keeps the columns whose importance is above the average. A forest asks its questions in combination, so it can keep a column that scores low alone.
- **Search** (`RFE`). Start with all columns: fit, drop the column with the smallest weight, and repeat until `n_features_to_select` are left. `SequentialFeatureSelector` goes the other way: start empty and add, one at a time, the column that raises the cross-validation score most. Thorough, but slow.

One line each, to copy:

```python
SelectKBest(f_classif, k=10)                                  # 1. score each column alone, keep the best 10
SelectFromModel(LinearSVC(penalty="l1", C=0.05, dual=False))  # 2. ask a linear model: nonzero weights
SelectFromModel(RandomForestClassifier(100, random_state=0))  # 2. ask a forest: importance above the average
RFE(LogisticRegression(), n_features_to_select=10)            # 3. search: drop the smallest weight, repeat
```

Whichever you choose, it goes **after the column transformer and before the model**, inside the pipeline, so that it is learned again in every fold (Lecture 7, "The leak"):

```python
prep_lin = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    verbose_feature_names_out=False)                   # keep the plain column names

my_select = make_pipeline(prep_lin, SelectKBest(f_classif), LogisticRegression(C=0.1))   # prepare, select, model
```

How many to keep is a setting: let a grid search choose it, then ask the best pipeline which columns it keeps. `best_estimator_` is already refitted on all of `X` with the best setting.

```python
grid_select = GridSearchCV(my_select, {"selectkbest__k": [5, 10, 20, 30, "all"]}, cv=5)
grid_select.fit(X, y)
print("best:", grid_select.best_params_, "  mean validation accuracy:", round(grid_select.best_score_, 3))

best = grid_select.best_estimator_
kept = best[0].get_feature_names_out()[best[1].get_support()]     # best[0]: preparation, best[1]: selection
print(len(kept), list(kept))
```

To try another selector, change the selection step and its grid together; the rest stays the same:

| Selection step | Grid |
|---|---|
| `SelectKBest(f_classif)` | `{"selectkbest__k": [5, 10, 20, 30, "all"]}` |
| `SelectFromModel(LinearSVC(penalty="l1", dual=False))` | `{"selectfrommodel__estimator__C": [0.01, 0.05, 0.1]}` |
| `SelectFromModel(RandomForestClassifier(100, random_state=0))` | no setting: run `cross_val_score`, then `.fit(X, y)` and read the kept columns |
| `RFE(LogisticRegression())` | `{"rfe__n_features_to_select": [5, 10, 20]}` |

To see a fixed number of columns, set it yourself: `my_select.set_params(selectkbest__k=10).fit(X, y)`, then read the kept columns from `my_select` in the same way.

### 5. Extra: gradient boosting

Many small trees in sequence, each fitted to the errors of the trees before it (Lecture 8, "Gradient boosting"). `HistGradientBoostingClassifier` even handles the gaps by itself, so it needs no imputer; it needs a dense table, hence `sparse_threshold=0`.

```python
prep_boost = make_column_transformer(
    ("passthrough", NUMERIC),                               # gaps and all
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    sparse_threshold=0)
my_boost = make_pipeline(prep_boost, HistGradientBoostingClassifier(random_state=0))
```

Its main settings: `learning_rate` (0.1 by default) and `max_iter`, the number of trees (100).

### All together: one pipeline from start to end

Every piece of the toolbox in one cell: prepare each group of columns, select, fit a model, and let a grid search choose the settings on the training set.

```python
other = [c for c in NUMERIC if c != "reviews_per_month"]
prep = make_column_transformer(                                     # 1. each group its own preparation
    (make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), ["reviews_per_month"]),
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), other),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
pipe = make_pipeline(prep, SelectKBest(f_classif), LogisticRegression(max_iter=1000))   # 2. prepare, select, model
grid = GridSearchCV(pipe, {"selectkbest__k": [10, 20, "all"],                           # 3. choose the settings
                           "logisticregression__C": [0.01, 0.1, 1]}, cv=5)
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 3))    # 4. the best mean validation accuracy: your estimate
```

For a forest, drop the scalers and the selection step, and change the model and the grid.

### Names in a grid search

A setting inside a pipeline is called by the name of its step in lower case, two underscores, and its own name: `logisticregression__C`, `decisiontreeclassifier__max_depth`, `randomforestclassifier__max_features`, `selectkbest__k`, `histgradientboostingclassifier__learning_rate`. A grid search can try several settings at once: `{"selectkbest__k": [10, 40], "logisticregression__C": [0.01, 0.1, 1]}` tries all six combinations.

</details>

### Step 1. The starting pipeline

**Run.** The pipeline of Lab 3, with one new step: a median imputer before the scaler. It is your starting point.

In [ ]:
# Step 1. Run this cell. Nothing to change.
my_pipe = make_pipeline(
    make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
    LogisticRegression(C=0.1))

scores = cross_val_score(my_pipe, X, y, cv=5)
print("Step 1, starting pipeline: mean validation accuracy", round(scores.mean(), 3))

*Stuck?* Toolbox, part 1 · Cheat sheet, section 2 "Filling the gaps" · Lecture 7, "The imputer belongs in the pipeline".

### Step 2. Fill the gaps your way

Task 1 showed that the gaps have reasons. Change the imputer of Step 1 and see what happens.

In [ ]:
# Step 2. Three ways to fill, and one with a separate fill for reviews_per_month.
zero_fill = ["reviews_per_month"]
other = [c for c in NUMERIC if c not in zero_fill]
candidates = {
    "median + indicator": make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
    "5 nearest neighbors": make_column_transformer(
        (make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5), StandardScaler()), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
    "0 for reviews per month, median + indicator for the rest": make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), zero_fill),
        (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), other),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
}
for name, prep in candidates.items():
    pipe = make_pipeline(prep, LogisticRegression(C=0.1))
    print(f"Step 2, {name}: mean validation accuracy", round(cross_val_score(pipe, X, y, cv=5).mean(), 3))

# All within a few thousandths of Step 1: with one listing in six having a gap, the fill
# hardly changes the score. We keep the fill that follows our knowledge of the gaps.

*Stuck?* Toolbox, part 1 · Cheat sheet, section 2 "Filling the gaps" · Lecture 7, "Fill in a constant" and "The absence is informative".

### Step 3. One tree against a forest

The best tree of Task 2 is in `grid_tree`. Does a forest of 200 trees beat it? Does it beat logistic regression? And which features does the forest use most?

In [ ]:
# Step 3. The best tree of Task 2 against a forest, with max_features chosen by a grid search.
print("Step 3, best tree:", grid_tree.best_params_, ": mean validation accuracy", round(grid_tree.best_score_, 3))

my_forest = make_pipeline(prep_tree, RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1))
print("Step 3, forest: mean validation accuracy", round(cross_val_score(my_forest, X, y, cv=5).mean(), 3))

my_forest.fit(X, y)
importances = pd.Series(my_forest[-1].feature_importances_, index=my_forest[0].get_feature_names_out())
print("the 10 most important features:")
print(importances.sort_values(ascending=False).head(10).round(3))

grid_forest = GridSearchCV(my_forest, {"randomforestclassifier__max_features": ["sqrt", 0.2, 0.4, None]}, cv=5)
grid_forest.fit(X, y)
print("Step 3, forest, best max_features =", grid_forest.best_params_["randomforestclassifier__max_features"],
      ": mean validation accuracy", round(grid_forest.best_score_, 3))
# The forest is far better than one tree, and about as good as logistic regression: no model
# wins everywhere. The default max_features ("sqrt") is as good as any other value here.
# The importances, read with doubt (Lecture 8, "Read with doubt"): they have no sign, so they do
# not say whether many reviews per month go with premium listings or with cheap ones; they say
# nothing about causes; and twins (guests, bedrooms, beds) split the credit between them.

*Stuck?* Toolbox, parts 2 and 3 · Cheat sheet, sections 5 "Decision trees" and 6 "Random forests" · Lecture 8, "Choosing the depth", "Random forest" and "Feature importances".

### Step 4. Choosing features

After one-hot encoding, your pipeline sees about 40 columns. Does it need all of them? Add a selection step to your logistic regression, and look at which columns it keeps.

In [ ]:
# Step 4. Choosing features for logistic regression: scores alone, then the forest.
prep_lin_names = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), zero_fill),
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), other),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    verbose_feature_names_out=False)
my_select = make_pipeline(prep_lin_names, SelectKBest(f_classif), LogisticRegression(C=0.1, max_iter=2000))
grid_select = GridSearchCV(my_select, {"selectkbest__k": [5, 10, 20, 30, "all"]}, cv=5)
grid_select.fit(X, y)
print("Step 4, best", grid_select.best_params_, ": mean validation accuracy", round(grid_select.best_score_, 3))

my_select.set_params(selectkbest__k=10).fit(X, y)
print("the 10 columns that score best alone:", list(my_select[0].get_feature_names_out()[my_select[1].get_support()]))

ask_forest = make_pipeline(prep_lin_names, SelectFromModel(RandomForestClassifier(100, random_state=0)),
                           LogisticRegression(C=0.1, max_iter=2000))
print("Step 4, the forest chooses: mean validation accuracy", round(cross_val_score(ask_forest, X, y, cv=5).mean(), 3))
ask_forest.fit(X, y)
print("the columns the forest keeps:", list(ask_forest[0].get_feature_names_out()[ask_forest[1].get_support()]))
# Keeping all 43 columns is best: with 1,500 listings and only 43 columns there is little to throw
# away. The 10 columns that score best alone are size, room type and reviews; km_to_center is not
# among them. Alone, the distance separates premium listings only weakly; together with size and
# room type (a whole home in the center) it matters much more. Scoring each feature alone misses
# this (Lecture 7, "Score each feature alone"). The forest asks its questions in combination, and
# keeps km_to_center.

*Stuck?* Toolbox, part 4 · Cheat sheet, section 3 "Choosing features" · Lecture 7, "Score each feature alone" and "How many to keep?".

### Your notes

**Your turn.** Double-click the cell below and write down, for each step, what you tried and its mean validation accuracy. Then choose your best pipeline.

**Answer.** Step 1, median imputer: about 0.78. Step 2, other fills: all within a few thousandths; we keep 0 for `reviews_per_month` and median + indicator for the rest, because it follows our knowledge of the gaps. Step 3, the best tree: about 0.74; the forest: about 0.78, and the default `max_features` is as good as any; its most important features are the distance, the availability, the reviews and the size. Step 4, selection: keeping all 43 columns is best, about 0.79; with the 10 that score best alone about 0.74, and `km_to_center` is not among them; the forest keeps 13 columns, `km_to_center` included, at about 0.78. Our choice: the forest of Step 3, which needs no scaling and little tuning, with an estimate of 0.78. Logistic regression would be just as good: the differences are the noise between folds.

### Your submission

**Your turn.** Fill in your pipeline, a few words on how you chose it, and the accuracy you expect on the locked listings (your best mean validation accuracy is a good estimate). The last line fits your pipeline once more on **all** 1,500 training listings.

In [ ]:
my_pipeline = grid_forest.best_estimator_     # Step 3: forest, max_features chosen by a grid search
my_strategy = "a random forest, max_features chosen by a 5-fold grid search"
my_estimate = round(grid_forest.best_score_, 2)

my_pipeline.fit(X, y)  # learn from all 1,500 training listings

In [ ]:
# Check
check_submission()

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare your pipeline with your neighbor's: the same model? The same fill?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it scores your pipeline on the 2,000 listings that no model has seen, gaps included.

In [ ]:
X_test, y_test = unlock_test_set("manytrees")    # the password on the screen

In [ ]:
test_accuracy = report(my_pipeline, my_estimate, X_test, y_test)

Tell the tutor your pipeline, how you chose it, your estimate and your test accuracy. We collect them on the screen.

> **As in Labs 2 and 3:** the test set has now been used. Changing your pipeline to do better on it would make its score an optimistic estimate.

## Task 6. What happened?

**Run.** The pieces of today, one by one: for each pipeline, the mean 5-fold cross-validation accuracy on your 1,500 listings, and the accuracy on the test set. This takes about a minute.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
from sklearn.dummy import DummyClassifier
lin = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
lin_ind = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
trees = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
candidates = {
    "always the most common class": DummyClassifier(),
    "logistic regression, median": make_pipeline(lin, LogisticRegression(C=0.1)),
    "  + missing indicators": make_pipeline(lin_ind, LogisticRegression(C=0.1)),
    "tree, depth 4": make_pipeline(trees, DecisionTreeClassifier(max_depth=4, random_state=0)),
    "tree, full depth": make_pipeline(trees, DecisionTreeClassifier(random_state=0)),
    "forest, 200 trees": make_pipeline(trees, RandomForestClassifier(200, random_state=0, n_jobs=-1)),
}
rows = []
for name, m in candidates.items():
    m.fit(X, y)
    rows.append({"pipeline": name,
                 "training": m.score(X, y),
                 "5-fold CV": cross_val_score(m, X, y, cv=5).mean(),
                 "test set": m.score(X_test, y_test)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) The full tree: compare its training accuracy with its test accuracy. What is going on, and which setting prevents it? (Lecture 8, "Keep the tree small")<br>
(b) The forest grows full trees too. Why does it not have the same problem? (Lecture 8, "Random forest")<br>
(c) Forest or logistic regression: which would you choose here, and why? (Lecture 8, "Which model when?")<br>
(d) Your forest of Step 3 ranks `reviews_per_month` among its most important features. Should a host collect more reviews to make the listing premium?<br>
(e) In Lab 3 we dropped the listings with gaps. What would that pipeline have done with the test listings that have a gap?

**Answer.** (a) The full tree gets (almost) every training listing right and does clearly worse on the test set: it memorizes the training data, noise included. That is overfitting. Limit its size, with `max_depth`, `min_samples_leaf` or `max_leaf_nodes`, chosen by cross-validation. (b) Every tree overfits in its own way: each sees a different bootstrap sample and chooses its questions among different features. Their votes average the individual mistakes away, so the forest is much more stable than any single tree. (c) Their accuracy is about the same here. Logistic regression is easier to read (signed coefficients) and gives probabilities; the forest needs no scaling and finds curves and interactions by itself. With equal scores, either is a sound choice; let cross-validation and the purpose decide. (d) No. An importance has no sign and says nothing about causes. The number of reviews per month says what kind of listing it is (how often it is booked, and by whom), not why it costs what it costs. The importance does not even say whether more reviews go with premium listings or with cheaper ones. (e) Nothing: a model that never saw a gap stops at a `NaN`. The pipeline would have had to skip those listings, about one in six, and they are not a random sample: new listings without reviews, private rooms without a minimum stay.

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** A listing has no review score because it has no reviews yet. Which kind of gap is this?

A. By accident &nbsp; B. It depends on another column &nbsp; C. It depends on the missing value itself &nbsp; D. It is not a gap

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Whether the score is missing follows from another column, `number_of_reviews` (Lecture 7, "Why is it missing?"). That column also tells you what the gap means.

</details>

**2.** A colleague chooses the 10 best columns with `SelectKBest` on all 1,500 listings, and then reports the 5-fold cross-validation accuracy of a model on those 10. The estimate is

A. unbiased &nbsp; B. too optimistic &nbsp; C. too pessimistic

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The selection looked at the labels of the validation folds too. Selection learns from the labels, so it belongs inside the pipeline (Lecture 7, "The leak").

</details>

**3.** Which of the following is **not** true for simple univariate feature selection?

A. It considers each feature on its own &nbsp; B. Univariate tests are often very fast to compute &nbsp; C. A feature is selected if it is only informative in combination with another feature &nbsp; D. It does not depend on the model you apply after the selection

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** A univariate test looks at one feature at a time, so it misses a feature that is only useful together with another one (Lecture 7).

</details>

**4.** A leaf of a tree holds 40 training listings, 30 of them premium. What does the tree predict for a new listing that lands there?

A. premium, with probability 0.75 &nbsp; B. not premium &nbsp; C. premium, with probability 1 &nbsp; D. It depends on the depth

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** The majority class of the leaf, with the share of that class as its probability: `predict_proba` gives [0.25, 0.75] (Lecture 8, "Trees in scikit-learn"). Its Gini impurity is 1 − 0.75² − 0.25² = 0.375.

</details>

**5.** Which of the following statements are true?

(a) k-NN imputation trains a k-nearest-neighbor classifier to fill in the missing values.<br>
(b) The main advantage of model-based feature selection is that it does not need labels.

<details><summary><b>Answer</b> (click to unfold)</summary>

Both are **false**. (a) k-NN imputation fills a gap with the average of the values of the nearest rows; no classifier is trained. (b) Model-based selection fits a supervised model, so it needs the labels (Lecture 7).

</details>

**6.** You measure the distance to the center in meters instead of kilometers. What happens to a decision tree?

A. It gets better &nbsp; B. It gets worse &nbsp; C. The same tree, with thresholds 1,000 times larger &nbsp; D. It needs a scaler now

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** A question compares one feature with a threshold; changing the unit only changes the threshold. Trees need no scaling (Lecture 8, "Trees in scikit-learn").

</details>

**7.** Which of the following is **not** a common way to limit the complexity of a decision tree and prevent overfitting?

A. Maximizing the purity of the leaves &nbsp; B. Limiting the maximum depth of the tree &nbsp; C. Limiting the maximum number of leaves &nbsp; D. Requiring a minimum number of points in a node to keep splitting it

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** A tree that keeps splitting until every leaf is pure is exactly the tree that overfits. B, C and D all stop the tree earlier (Lecture 8).

</details>

**8.** A random forest is a collection of decision trees whose predictions are combined. Which of the following statements about these trees are true?

(a) Each tree is built on a sample of the data.<br>
(b) When a tree is built, only a random subset of the features is considered at each split.

<details><summary><b>Answer</b> (click to unfold)</summary>

Both are **true**: (a) each tree gets its own bootstrap sample of the rows, and (b) at each split it may only choose among `max_features` randomly chosen features. Both make the trees different from each other (Lecture 8).

</details>

**9.** You double the number of trees in a random forest, from 200 to 400. What happens?

A. It overfits more &nbsp; B. It costs more time, and the accuracy stays about the same or improves a little &nbsp; C. It needs a smaller `max_depth` &nbsp; D. It becomes a gradient boosting model

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** More trees never hurt, they only cost time (Lecture 8, "How many trees? How many features per question?").

</details>

**10.** Which of the following statements about random forests are true?

(a) A high `max_features` means that the trees in the forest will be quite different.<br>
(b) For the parameter `n_estimators`, larger is better.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) **False:** with more features to choose from, the trees pick the same strong features and look more alike. (b) **True:** more trees never hurt the accuracy; they only cost time (Lecture 8).

</details>

## If you finish early

**A. The leak, by hand.** Add 1,000 columns of pure noise to the 12 numbers, and give every listing a random label. There is nothing to learn. Choose the 10 best columns with `SelectKBest` on all listings first, then cross-validate a logistic regression on them; then do the same with the selection inside the pipeline. Compare the two accuracies with 0.5 (Lecture 7, "The leak").

In [ ]:
rng = np.random.default_rng(0)
noise = rng.normal(size=(len(X), 1000))      # 1,000 columns of noise
coin = rng.integers(0, 2, len(X))            # labels tossed by a coin

selected = SelectKBest(f_classif, k=10).fit_transform(noise, coin)       # selection on everything
print("selection outside the pipeline:", round(cross_val_score(LogisticRegression(), selected, coin, cv=5).mean(), 3))
inside = make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression())
print("selection inside the pipeline: ", round(cross_val_score(inside, noise, coin, cv=5).mean(), 3))
# Outside: clearly above 0.5 (about 0.57), although there is nothing to learn. Inside: about 0.5,
# the honest answer. With fewer listings and more noise columns, the gap grows (Lecture 7: 0.79).

**B. A tree cannot extrapolate.** Fit `DecisionTreeRegressor(max_depth=3)` on the number of guests alone, with `y` as a number: in every leaf it predicts the share of premium listings. Predict for 1, 4, 8, 16 and 32 guests. What happens beyond the largest listing in the training data? (Lecture 8, "Trees for regression".)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
reg = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X[["accommodates"]], y)
new = pd.DataFrame({"accommodates": [1, 4, 8, 16, 32]})
print(pd.DataFrame({"guests": new["accommodates"], "predicted share premium": reg.predict(new).round(2)}))
# From the largest number of guests in the training data on, the prediction stays the same:
# the last leaf goes on forever. A tree never predicts outside the range it saw in training.

## Where this is going

> Every step that learns from data, the imputer and the selection included, goes inside the pipeline. Then every listing gets a prediction, and cross-validation stays honest.

Today the gaps had reasons, and a forest of trees matched the linear model of Lab 3 without any scaling or tuning of features. No model wins everywhere: start simple, and let cross-validation decide.

**Coming up.**

- Lecture 9 (Monday, November 23): model evaluation. Accuracy has been our only yardstick so far; it is not always the right one.
- Lecture 10 (Tuesday, November 24): imbalanced data.

**Reading.** Müller and Guido (2016), Chapter 2: "Decision Trees" and "Ensembles of Decision Trees"; Chapter 4: "Automatic Feature Selection". Imputation: scikit-learn User Guide, "Imputation of missing values".

---
*This lab practices Lectures 7 and 8 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1, without dropping the listings with gaps.*